<a href="https://colab.research.google.com/github/poojitha2606/gen-ai-experiments/blob/main/exp_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu transformers datasets evaluate -q
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import textwrap
documents = [
 """Machine learning is a subset of artificial intelligence that enables systems
 to learn and improve from experience without being explicitly programmed.""",
 """Deep learning is a branch of machine learning that uses neural networks
 with many layers to analyze various types of data.""",
 """Natural Language Processing (NLP) allows computers to understand, interpret
 and generate human language.""",
 """Retrieval-Augmented Generation (RAG) combines information retrieval with
 text generation to improve the accuracy of responses.""",
 """Transformers are deep learning models that use attention mechanisms and
 are widely used in NLP tasks such as translation and text generation.""",
 """FAISS is a library developed by Facebook for efficient similarity search
 and clustering of dense vectors."""
]
print(f"Loaded {len(documents)} documents.\n")
def chunk_text(text, chunk_size=50):
 words = text.split()
 chunks = []
 for i in range(0, len(words), chunk_size):
 chunk = " ".join(words[i:i+chunk_size])
 chunks.append(chunk)
 return chunks
all_chunks = []
for doc in documents:
 all_chunks.extend(chunk_text(doc))
print(f"Total Chunks Created: {len(all_chunks)}\n")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Generating embeddings for all chunks...\n")
chunk_embeddings = embedder.encode(all_chunks)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings))
print(f"FAISS Index Created with {index.ntotal} vectors.\n")
generator = pipeline(
 "text-generation",
 model="gpt2",
 device=0 if torch.cuda.is_available() else -1
)
print("Generator model loaded successfully!\n")
def rag_pipeline(query, top_k=3):
 print(f"\nQUERY: {query}\n")
 query_embedding = embedder.encode([query])
 distances, indices = index.search(np.array(query_embedding), top_k)
 retrieved_chunks = [all_chunks[i] for i in indices[0]]
 print("=== Retrieved Context ===")
 for i, chunk in enumerate(retrieved_chunks):
 print(f"\nChunk {i+1}:")
 print(textwrap.fill(chunk, width=80))
 context = " ".join(retrieved_chunks)
 prompt = f"""
 Use the following context to answer the question:
 Context:
 {context}
 Question: {query}
 Answer:
 """
 output = generator(
 prompt,
 max_new_tokens=120,
 temperature=0.7,
 top_p=0.9
 )
 answer = output[0]['generated_text']
 print("\n=== Generated Answer ===\n")
 print(textwrap.fill(answer, width=100))
 return answer
queries = [
 "What is RAG in AI?",
 "Explain machine learning.",
 "What are transformers?",
 "What is FAISS used for?"
]
answers = []
for q in queries:
 ans = rag_pipeline(q)
 answers.append(ans)
 print("\n" + "="*120)
from sentence_transformers import util
reference_answers = [
 "RAG combines retrieval and generation to improve accuracy.",
 "Machine learning allows systems to learn from data.",
 "Transformers use attention mechanisms.",
 "FAISS is used for similarity search."
]
print("\n=== EVALUATION RESULTS ===\n")
for i in range(len(queries)):
 emb1 = embedder.encode(reference_answers[i], convert_to_tensor=True)
 emb2 = embedder.encode(answers[i], convert_to_tensor=True)
 similarity = util.cos_sim(emb1, emb2).item()
 print(f"Query {i+1}: {queries[i]}")
 print(f"Semantic Similarity Score: {similarity:.4f}")
 print("-"*60)